# Manufacturing Supply Chain Analysis
## Step 1: Data Quality Audit

**Author:** Hiten Shah  
**Data:** Six months (Jan–Jun 2026) of Purchase, Store Inventory and Sales records from a wear-parts foundry. Company, vendor, customer and sales-rep names are anonymised and monetary values rescaled.

Nine checks are run across the three sheets. Every issue is logged with its evidence before anything is changed, and the cleaned tables are exported for the SQL analysis in Step 2.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

FILE = Path('../data/raw/case_study_datasets.xlsx')

purchase = pd.read_excel(FILE, sheet_name='Purchase')
store    = pd.read_excel(FILE, sheet_name='Store_Inventory')
sales    = pd.read_excel(FILE, sheet_name='Sales')

print(purchase.shape, store.shape, sales.shape)

(131, 13) (1201, 11) (407, 14)


### The issue log

Every problem found is recorded here rather than fixed silently, so each
correction can be traced back to its evidence.

In [2]:
issues = []

def log(sheet, check, severity, count, description, action):
    """Record one data quality issue."""
    issues.append({'Sheet': sheet,
                   'Check': check,
                   'Severity': severity,
                   'Records': count,
                   'Issue': description,
                   'Action taken': action})

print("Issue log ready.")

Issue log ready.


### Check 1 — Structural profile

Confirm each sheet covers the stated Jan–Jun 2026 window and that date
columns loaded as dates, not text.

In [3]:
for name, df, datecol in [('Purchase', purchase, 'PO_Date'),
                          ('Store_Inventory', store, 'Txn_Date'),
                          ('Sales', sales, 'Order_Date')]:
    print(f"--- {name} ---")
    print(f"  rows       : {len(df)}")
    print(f"  date range : {df[datecol].min().date()}  ->  {df[datecol].max().date()}")
    print(f"  date dtype : {df[datecol].dtype}")
    print()

--- Purchase ---
  rows       : 131
  date range : 2026-01-01  ->  2026-06-30
  date dtype : datetime64[us]

--- Store_Inventory ---
  rows       : 1201
  date range : 2026-01-02  ->  2026-07-05
  date dtype : datetime64[us]

--- Sales ---
  rows       : 407
  date range : 2026-01-01  ->  2026-06-30
  date dtype : datetime64[us]



In [4]:
store_july = store[store.Txn_Date > '2026-06-30']

print(f"Store transactions after 30-Jun: {len(store_july)}\n")
print(store_july[['Txn_ID','Txn_Date','Item_Name','Movement_Type','Quantity']].to_string(index=False))

Store transactions after 30-Jun: 6

  Txn_ID   Txn_Date         Item_Name     Movement_Type  Quantity
ST-00905 2026-07-01     Crusher Liner Finished Goods-In        11
ST-00906 2026-07-01     Crusher Liner Finished Goods-In        12
ST-00907 2026-07-03     Crusher Liner Finished Goods-In        11
ST-01064 2026-07-04 Hammer - Standard Finished Goods-In        27
ST-01065 2026-07-05 Hammer - Standard Finished Goods-In        38
ST-01201 2026-07-03        Wear Plate Finished Goods-In        39


In [5]:
log('Store_Inventory', 'Date range', 'Info', len(store_july),
    'Ledger extends to 05-Jul-2026, past the Jan-Jun window; all 6 rows are '
    'Finished Goods-In (production completions), not customer movements',
    'Retained - the store extract simply ran a few days past the Sales/Purchase cut-off')

print(f"Logged. Issues recorded so far: {len(issues)}")

Logged. Issues recorded so far: 1


### Check 2 — Duplicates

Two kinds: exact repeated rows, and rows with a new ID describing the same
real transaction. The second is the dangerous one, because the unique ID
makes it look legitimate.

In [6]:
for name, df, idcol in [('Purchase', purchase, 'PO_ID'),
                        ('Store_Inventory', store, 'Txn_ID'),
                        ('Sales', sales, 'Order_ID')]:
    print(f"{name:<16} exact duplicate rows: {df.duplicated().sum():<3} "
          f"| repeated {idcol}: {df[idcol].duplicated().sum()}")

Purchase         exact duplicate rows: 0   | repeated PO_ID: 0
Store_Inventory  exact duplicate rows: 0   | repeated Txn_ID: 0
Sales            exact duplicate rows: 0   | repeated Order_ID: 0


In [7]:
BUSINESS_KEY = ['Order_Date','Customer_Name','Product_SKU','Quantity_Units','Unit_Price_INR']

dupes = sales[sales.duplicated(BUSINESS_KEY, keep=False)].sort_values(BUSINESS_KEY + ['Order_ID'])

print(f"Rows sharing a business key: {len(dupes)}\n")
print(dupes[['Order_ID','Order_Date','Customer_Name','Product_SKU',
             'Quantity_Units','Total_Value_INR','Order_Status']].to_string(index=False))

Rows sharing a business key: 12

Order_ID Order_Date               Customer_Name Product_SKU  Quantity_Units  Total_Value_INR Order_Status
 SO-0088 2026-01-01            Summit Cement Ld    SKU-ML02              22        183505.52    Fulfilled
 SO-0405 2026-01-01            Summit Cement Ld    SKU-ML02              22        183505.52    Fulfilled
 SO-0261 2026-04-08               Midland Mines    SKU-HM02              28        123757.20    Fulfilled
 SO-0402 2026-04-08               Midland Mines    SKU-HM02              28        123757.20    Fulfilled
 SO-0272 2026-04-09               Midland Mines    SKU-CR01              27        188849.34    Fulfilled
 SO-0406 2026-04-09               Midland Mines    SKU-CR01              27        188849.34    Fulfilled
 SO-0078 2026-04-15 Riverbend Cooperative Sugar    SKU-BS01              11         32251.78    Fulfilled
 SO-0403 2026-04-15 Riverbend Cooperative Sugar    SKU-BS01              11         32251.78    Fulfilled
 SO-0285 2026

In [8]:
extra_rows = len(dupes) - dupes.drop_duplicates(BUSINESS_KEY).shape[0]
inflated_by = dupes.Total_Value_INR.sum() - dupes.drop_duplicates(BUSINESS_KEY).Total_Value_INR.sum()

print(f"Duplicate order rows     : {extra_rows}")
print(f"Revenue overstated by    : Rs {inflated_by:,.0f}")
print(f"As % of reported revenue : {inflated_by / sales.Total_Value_INR.sum() * 100:.2f}%")

Duplicate order rows     : 6
Revenue overstated by    : Rs 892,692
As % of reported revenue : 2.11%


In [9]:
log('Sales', 'Duplicate orders', 'High', extra_rows,
    f'{extra_rows} orders entered twice under new IDs (IDs SO-0402 to SO-0407 appear in every pair), '
    f'pointing to a batch re-import; revenue overstated by Rs {inflated_by:,.0f}',
    'Kept the first occurrence of each pair, dropped the second')

sales_dedup = sales.drop_duplicates(BUSINESS_KEY, keep='first').copy()

print(f"Sales rows: {len(sales)} -> {len(sales_dedup)}")
print(f"Issues recorded so far: {len(issues)}")

Sales rows: 407 -> 401
Issues recorded so far: 2


### Check 3 — Inconsistent naming

One customer entered under two spellings is counted as two customers,
which distorts every ranking and total by customer.

In [10]:
print(f"Distinct customer names as entered: {sales.Customer_Name.nunique()}\n")

for n in sorted(sales.Customer_Name.unique()):
    print(" ", n)

Distinct customer names as entered: 20

  Eastern Ore & Minerals
  Granite Ridge Mining
  Granite Ridge Mining.
  Greenfield Sugar Works
  Greenfield Sugars Works
  Harbour Cement
  Highland Minerals
  Keystone Cement Corp
  Lakeside Power Station
  Midland Mines
  Northgate Thermal Power
  Ridgeline Cement Works
  Riverbend Cooperative Sugar
  Southern Power Corp
  Summit Cement Ld
  Summit Cement Ltd
  Sunrise Sugars
  Sunrize Sugars
  Valley Sugar Mills
  Westfield Power Plant


In [11]:
NAME_FIX = {
    'Granite Ridge Mining.'   : 'Granite Ridge Mining',
    'Summit Cement Ld'        : 'Summit Cement Ltd',
    'Greenfield Sugars Works' : 'Greenfield Sugar Works',
    'Sunrize Sugars'          : 'Sunrise Sugars',
}

check = sales_dedup[sales_dedup.Customer_Name.isin(list(NAME_FIX) + list(NAME_FIX.values()))]

print(check.groupby(['Customer_Name','Customer_Industry','Region'])
           .agg(orders=('Order_ID','count'), value=('Total_Value_INR','sum'))
           .round(0).to_string())

                                                       orders      value
Customer_Name           Customer_Industry Region                        
Granite Ridge Mining    Mining            Karnataka        25  2889334.0
Granite Ridge Mining.   Mining            Karnataka         6   581891.0
Greenfield Sugar Works  Sugar             Maharashtra      18  1300226.0
Greenfield Sugars Works Sugar             Maharashtra       6   602533.0
Summit Cement Ld        Cement            Maharashtra      11  1162125.0
Summit Cement Ltd       Cement            Maharashtra      21  3023900.0
Sunrise Sugars          Sugar             Maharashtra      11   753923.0
Sunrize Sugars          Sugar             Maharashtra       6   647385.0


In [12]:
before = (sales_dedup.groupby('Customer_Name').Total_Value_INR.sum()
                     .sort_values(ascending=False).head(8).round(0))

sales_fixed = sales_dedup.copy()
sales_fixed['Customer_Name'] = sales_fixed.Customer_Name.replace(NAME_FIX)

after = (sales_fixed.groupby('Customer_Name').Total_Value_INR.sum()
                    .sort_values(ascending=False).head(8).round(0))

print(pd.DataFrame({'Before cleanup': before, 'After cleanup': after}).to_string())

print(f"\nDistinct customers: {sales_dedup.Customer_Name.nunique()} entered "
      f"-> {sales_fixed.Customer_Name.nunique()} actual")

                        Before cleanup  After cleanup
Customer_Name                                        
Eastern Ore & Minerals       2223019.0      2223019.0
Granite Ridge Mining         2889334.0      3471226.0
Harbour Cement               5562172.0      5562172.0
Highland Minerals            3424095.0      3424095.0
Keystone Cement Corp         4119156.0      4119156.0
Midland Mines                2377061.0      2377061.0
Ridgeline Cement Works       3295218.0      3295218.0
Summit Cement Ltd            3023900.0      4186025.0

Distinct customers: 20 entered -> 16 actual


In [13]:
# How much did the split spelling distort the biggest affected customer?
key = 'Summit Cement Ltd'
rank_before = list(before.index).index(key) + 1
rank_after  = list(after.index).index(key) + 1

log('Sales', 'Inconsistent customer names', 'High', len(check),
    f'{len(NAME_FIX)} customers entered under two spellings each (e.g. "Summit Cement Ld" / '
    '"Summit Cement Ltd"); 20 names represent 16 real customers. Each pair confirmed as one '
    'customer by matching industry and region',
    f'Standardised to the majority spelling; {key} rises from Rs {before[key]/1e5:.1f}L to '
    f'Rs {after[key]/1e5:.1f}L and from #{rank_before} to #{rank_after} customer by value')

print(f"{key}: Rs {before[key]:,.0f} (#{rank_before}) -> Rs {after[key]:,.0f} (#{rank_after})")
print(f"Issues recorded so far: {len(issues)}")

Summit Cement Ltd: Rs 3,023,900 (#5) -> Rs 4,186,025 (#2)
Issues recorded so far: 3


In [14]:
print(f"Distinct vendor names: {purchase.Vendor_Name.nunique()}\n")
for v in sorted(purchase.Vendor_Name.unique()):
    print(f"  {v:<25} ({purchase[purchase.Vendor_Name==v].Vendor_City.iloc[0]})")

Distinct vendor names: 6

  Central Ferro Alloys      (Nagpur)
  Coastal Steel Traders     (Mumbai)
  Eastline Metal Works      (Nagpur)
  Hilltop Scrap & Metals    (Kolhapur)
  Meridian Ferro Industries (Raipur)
  Plateau Alloys Pvt Ltd    (Pune)


### Check 4 — Missing values

A blank is not automatically an error. Each one is checked against the
surrounding data to decide whether it is a genuine gap or a correct
reflection of business reality.

In [15]:
for name, df in [('Purchase', purchase), ('Store_Inventory', store), ('Sales', sales_dedup)]:
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f"--- {name} ---")
    print(nulls.to_string() if len(nulls) else "  (none)")
    print()

--- Purchase ---
Quality_Grade           2
Actual_Delivery_Date    7

--- Store_Inventory ---
Linked_PO_ID             1074
Linked_Sales_Order_ID     823

--- Sales ---
Actual_Dispatch_Date    29



In [16]:
open_pos = purchase[purchase.Actual_Delivery_Date.isna()]

print(open_pos[['PO_ID','PO_Date','Vendor_Name','Material_Name',
                'Quantity_Kg','Promised_Lead_Time_Days']].to_string(index=False))

print(f"\nAll placed in June?  {(open_pos.PO_Date.dt.month == 6).all()}")
print(f"Earliest of these    : {open_pos.PO_Date.min().date()}")
print(f"Goods receipts in store ledger referencing these POs: "
      f"{store[store.Linked_PO_ID.isin(open_pos.PO_ID)].shape[0]}")

  PO_ID    PO_Date           Vendor_Name   Material_Name  Quantity_Kg  Promised_Lead_Time_Days
PO-0016 2026-06-28 Coastal Steel Traders     Steel Scrap         3335                       10
PO-0033 2026-06-24 Coastal Steel Traders       MS Plates         1162                       12
PO-0065 2026-06-26  Central Ferro Alloys Ferro Manganese          599                       10
PO-0114 2026-06-18  Eastline Metal Works        Pig Iron         1459                       13
PO-0115 2026-06-30  Eastline Metal Works        Pig Iron         1576                       13
PO-0130 2026-06-17  Eastline Metal Works Cast Iron Scrap         1808                       11
PO-0131 2026-06-25  Eastline Metal Works Cast Iron Scrap         1509                       13

All placed in June?  True
Earliest of these    : 2026-06-17
Goods receipts in store ledger referencing these POs: 0


In [17]:
log('Purchase', 'Missing delivery dates', 'Info', len(open_pos),
    '7 POs have no Actual_Delivery_Date - all placed 17-30 June with lead times of 10-13 days, '
    'and no goods receipt exists in the store ledger for any of them',
    'Classified as open POs in transit at the cut-off; excluded from lead-time analysis '
    'rather than counted as late deliveries')

print(f"Issues recorded so far: {len(issues)}")

Issues recorded so far: 4


In [18]:
missing_grade = purchase[purchase.Quality_Grade.isna()]
print("POs with no quality grade:\n")
print(missing_grade[['PO_ID','PO_Date','Vendor_Name','Material_Name','Quantity_Kg']].to_string(index=False))

log('Purchase', 'Missing quality grade', 'Low', len(missing_grade),
    'Quality_Grade blank on 2 POs',
    'Left blank rather than guessed; flagged for the Purchase team to complete')

print("\n--- Sales: 29 missing dispatch dates ---")
print(pd.crosstab(sales_dedup.Order_Status, sales_dedup.Actual_Dispatch_Date.isna(),
                  rownames=['Order_Status'], colnames=['Dispatch date missing']).to_string())

pending = sales_dedup[sales_dedup.Order_Status == 'Pending']
print(f"\nPending orders were placed in: "
      f"{sorted(pending.Order_Date.dt.to_period('M').unique().astype(str))}")

POs with no quality grade:

  PO_ID    PO_Date               Vendor_Name Material_Name  Quantity_Kg
PO-0014 2026-06-01    Hilltop Scrap & Metals   Steel Scrap         1681
PO-0037 2026-02-06 Meridian Ferro Industries  Ferro Chrome         1436

--- Sales: 29 missing dispatch dates ---


Dispatch date missing  False  True 
Order_Status                       
Cancelled                  0      2
Delayed                   53      0
Fulfilled                314      0
Pending                    0     27
Returned-Rework            5      0

Pending orders were placed in: ['2026-06']


In [19]:
log('Sales', 'Missing dispatch dates', 'Info', 29,
    '29 orders have no Actual_Dispatch_Date - all are Pending (27, every one placed in June) '
    'or Cancelled (2); no Fulfilled order is missing a date',
    'Correct as-is; these rows are excluded from on-time delivery calculations')

print(f"Issues recorded so far: {len(issues)}")

Issues recorded so far: 6


### Checks 5-7 — Arithmetic, date logic, and invalid quantities

Three structural tests: do the totals recompute, are any dates impossible,
and does any stock movement have a nonsensical quantity?

In [20]:
# Check 5 - do the stored totals match Quantity x Rate?
p_bad = ((purchase.Total_Cost_INR - (purchase.Quantity_Kg * purchase.Unit_Cost_INR_per_Kg)).abs() > 1).sum()
s_bad = ((sales_dedup.Total_Value_INR - (sales_dedup.Quantity_Units * sales_dedup.Unit_Price_INR)).abs() > 1).sum()
print(f"CHECK 5  Purchase total mismatches : {p_bad}")
print(f"         Sales total mismatches    : {s_bad}")

# Check 6 - impossible date sequences
purchase['actual_lead_days'] = (purchase.Actual_Delivery_Date - purchase.PO_Date).dt.days
sales_dedup['dispatch_lag']  = (sales_dedup.Actual_Dispatch_Date - sales_dedup.Order_Date).dt.days
print(f"\nCHECK 6  Deliveries before the PO date   : {(purchase.actual_lead_days < 0).sum()}")
print(f"         Dispatches before the order date: {(sales_dedup.dispatch_lag < 0).sum()}")
print(f"         Actual lead time range          : "
      f"{purchase.actual_lead_days.min():.0f} to {purchase.actual_lead_days.max():.0f} days "
      f"(median {purchase.actual_lead_days.median():.0f})")

# Check 7 - invalid stock movements
bad_qty = store[store.Quantity <= 0]
print(f"\nCHECK 7  Store movements with quantity <= 0 : {len(bad_qty)}\n")
print(bad_qty[['Txn_ID','Txn_Date','Item_Code','Item_Name','Movement_Type',
               'Quantity','Balance_After_Txn']].to_string(index=False))

log('Purchase + Sales', 'Arithmetic and date logic', 'Info', 0,
    'Total = Quantity x Rate verified on all rows; no deliveries before PO date, '
    'no dispatches before order date',
    'No action needed')

CHECK 5  Purchase total mismatches : 0
         Sales total mismatches    : 0

CHECK 6  Deliveries before the PO date   : 0
         Dispatches before the order date: 0


         Actual lead time range          : 7 to 38 days (median 12)

CHECK 7  Store movements with quantity <= 0 : 4

  Txn_ID   Txn_Date Item_Code                    Item_Name        Movement_Type  Quantity  Balance_After_Txn
ST-00003 2026-01-10     RM-SS                  Steel Scrap Issued-to-Production         0                976
ST-00187 2026-04-19     RM-FM              Ferro Manganese Issued-to-Production       -62                706
ST-00248 2026-05-13     RM-FS                Ferro Silicon Issued-to-Production         0               1362
ST-00494 2026-03-29  SKU-GR02 Grinding Roller - Heavy Duty    Finished Goods-In       -10                 54


### Check 8 — Ledger balance reconciliation

The ledger's running-balance column is an independent record of every
movement. Reconciling quantity against balance identifies wrong entries —
and, where the balance column proves reliable, lets us recover the
correct values rather than discard the rows.

In [21]:
# Receipts increase stock; issues, dispatches and returns reduce it.
SIGN = {
    'Inward-Purchase'            :  1,
    'Finished Goods-In'          :  1,
    'Issued-to-Production'       : -1,
    'Dispatch-to-Sales'          : -1,
    'QC Reject-Return to Vendor' : -1,
}

led = store.sort_values(['Item_Code','Txn_Date','Txn_ID']).copy()
led['signed_qty']   = led.Quantity * led.Movement_Type.map(SIGN)
led['prev_balance'] = led.groupby('Item_Code').Balance_After_Txn.shift(1)
led['expected_balance'] = led.prev_balance + led.signed_qty
led['mismatch'] = led.expected_balance.notna() & (led.expected_balance != led.Balance_After_Txn)

checkable = led.prev_balance.notna().sum()
matched   = (~led.mismatch & led.prev_balance.notna()).sum()

print(f"Ledger rows checked        : {checkable}")
print(f"Rows that reconcile        : {matched}")
print(f"Rows that DO NOT reconcile : {led.mismatch.sum()}")
print(f"Reconciliation rate        : {matched / checkable * 100:.2f}%")

Ledger rows checked        : 1183
Rows that reconcile        : 1179
Rows that DO NOT reconcile : 4
Reconciliation rate        : 99.66%


In [22]:
broken = led[led.mismatch].copy()
broken['true_signed_qty'] = broken.Balance_After_Txn - broken.prev_balance
broken['true_quantity']   = broken.true_signed_qty.abs()

print(broken[['Txn_ID','Txn_Date','Item_Name','Movement_Type',
              'Quantity','prev_balance','Balance_After_Txn','true_quantity']].to_string(index=False))

  Txn_ID   Txn_Date                    Item_Name        Movement_Type  Quantity  prev_balance  Balance_After_Txn  true_quantity
ST-00187 2026-04-19              Ferro Manganese Issued-to-Production       -62         768.0                706           62.0
ST-00248 2026-05-13                Ferro Silicon Issued-to-Production         0        1595.0               1362          233.0
ST-00003 2026-01-10                  Steel Scrap Issued-to-Production         0        2129.0                976         1153.0
ST-00494 2026-03-29 Grinding Roller - Heavy Duty    Finished Goods-In       -10          44.0                 54           10.0


In [23]:
log('Store_Inventory', 'Ledger balance reconciliation', 'High', len(broken),
    '4 of 1,183 rows fail the running-balance check: 2 sign reversals (ST-00187, ST-00494) '
    'and 2 quantities that did not save (ST-00003, ST-00248). The other 99.66% reconcile, '
    'confirming Balance_After_Txn as the reliable source',
    'Quantities recovered from the balance movement rather than deleting the rows. '
    'Stock positions were never wrong - only the transaction records')

store_clean = store.copy()
for _, r in broken.iterrows():
    store_clean.loc[store_clean.Txn_ID == r.Txn_ID, 'Quantity'] = r.true_quantity

# Re-run the same reconciliation on the corrected data
chk = store_clean.sort_values(['Item_Code','Txn_Date','Txn_ID']).copy()
chk['signed'] = chk.Quantity * chk.Movement_Type.map(SIGN)
chk['prev']   = chk.groupby('Item_Code').Balance_After_Txn.shift(1)
remaining = (chk.prev.notna() & (chk.prev + chk.signed != chk.Balance_After_Txn)).sum()

print(f"Corrections applied. Remaining mismatches: {remaining}")
print(f"Issues recorded so far: {len(issues)}")

Corrections applied. Remaining mismatches: 0
Issues recorded so far: 8


### Check 9 — Referential integrity and status reliability

Do the join keys resolve? And does Order_Status actually mean what it
appears to mean when tested against the dates?

In [24]:
po_links = store.Linked_PO_ID.dropna()
so_links = store.Linked_Sales_Order_ID.dropna()

print(f"Store -> Purchase : {len(po_links)} links, {(~po_links.isin(purchase.PO_ID)).sum()} broken")
print(f"Store -> Sales    : {len(so_links)} links, {(~so_links.isin(sales.Order_ID)).sum()} broken")

print(f"\nInward-Purchase rows without a PO link  : "
      f"{store[(store.Movement_Type=='Inward-Purchase') & store.Linked_PO_ID.isna()].shape[0]}")
print(f"Dispatch rows without a sales order link : "
      f"{store[(store.Movement_Type=='Dispatch-to-Sales') & store.Linked_Sales_Order_ID.isna()].shape[0]}")

log('Store_Inventory', 'Referential integrity', 'Info', 0,
    'All PO and sales order links resolve to existing records; every inward receipt has a PO '
    'link and every dispatch has a sales order link',
    'No action needed - the three sheets join cleanly')

Store -> Purchase : 127 links, 0 broken


Store -> Sales    : 378 links, 0 broken

Inward-Purchase rows without a PO link  : 0
Dispatch rows without a sales order link : 0


In [25]:
dispatched = sales_dedup[sales_dedup.Actual_Dispatch_Date.notna()].copy()
dispatched['days_late'] = (dispatched.Actual_Dispatch_Date - dispatched.Promised_Dispatch_Date).dt.days

flagged  = (dispatched.Order_Status == 'Delayed').sum()
actually = (dispatched.days_late > 0).sum()

print(f"Dispatched orders                       : {len(dispatched)}")
print(f"Flagged 'Delayed' in Order_Status        : {flagged}  ({flagged/len(dispatched)*100:.1f}%)")
print(f"Actually dispatched after promised date  : {actually}  ({actually/len(dispatched)*100:.1f}%)")
print(f"\nOn-time delivery by the status field : {(1-flagged/len(dispatched))*100:.1f}%")
print(f"On-time delivery by actual dates     : {(1-actually/len(dispatched))*100:.1f}%")

hidden = dispatched[(dispatched.Order_Status=='Fulfilled') & (dispatched.days_late>0)]
print(f"\nOrders marked 'Fulfilled' but dispatched late: {len(hidden)}")
print(hidden.days_late.value_counts().sort_index().to_string())

Dispatched orders                       : 372
Flagged 'Delayed' in Order_Status        : 53  (14.2%)
Actually dispatched after promised date  : 219  (58.9%)

On-time delivery by the status field : 85.8%
On-time delivery by actual dates     : 41.1%

Orders marked 'Fulfilled' but dispatched late: 161
days_late
1    56
2    51
3    54


In [26]:
log('Sales', 'Order_Status reliability', 'Medium', len(hidden),
    f'{len(hidden)} orders marked "Fulfilled" were dispatched 1-3 days after the promised date. '
    'On-time delivery reads 85.8% from the status field but 41.1% measured from actual dates',
    'On-time performance recalculated from dates rather than the status field. Status retained '
    'to separate severe delays (15-30 days) from routine 1-3 day slippage')

print(f"Issues recorded so far: {len(issues)}")

Issues recorded so far: 10


## Building the cleaned dataset

Every correction traces back to a check above. Note that row counts barely
change — the data was corrected, not deleted. Only the 6 genuine duplicates
were removed.

In [27]:
# Purchase: flag open POs, compute lead time performance
purchase_clean = purchase.copy()
purchase_clean['Is_Open_PO'] = purchase_clean.Actual_Delivery_Date.isna()
purchase_clean['Actual_Lead_Time_Days'] = (purchase_clean.Actual_Delivery_Date - purchase_clean.PO_Date).dt.days
purchase_clean['Lead_Time_Slip_Days'] = purchase_clean.Actual_Lead_Time_Days - purchase_clean.Promised_Lead_Time_Days
purchase_clean['Delivered_Late'] = purchase_clean.Lead_Time_Slip_Days > 0

# Store: corrections already applied in Check 8
store_clean['Signed_Quantity'] = store_clean.Quantity * store_clean.Movement_Type.map(SIGN)

# Sales: duplicates already removed, now standardise the customer names
sales_clean = sales_dedup.copy()
sales_clean['Customer_Name'] = sales_clean.Customer_Name.replace(NAME_FIX)
sales_clean['Days_Late'] = (sales_clean.Actual_Dispatch_Date - sales_clean.Promised_Dispatch_Date).dt.days
sales_clean['Dispatched_Late'] = sales_clean.Days_Late > 0

print("CLEANED DATASET")
print("="*55)
print(f"Purchase : {len(purchase):>5} -> {len(purchase_clean):>5} rows ({purchase_clean.Is_Open_PO.sum()} open POs flagged)")
print(f"Store    : {len(store):>5} -> {len(store_clean):>5} rows ({len(broken)} quantities corrected)")
print(f"Sales    : {len(sales):>5} -> {len(sales_clean):>5} rows ({len(sales)-len(sales_clean)} duplicates removed)")
print()
print(f"Revenue as reported : Rs {sales.Total_Value_INR.sum():,.0f}")
print(f"Revenue after clean : Rs {sales_clean.Total_Value_INR.sum():,.0f}")
print(f"Difference          : Rs {sales.Total_Value_INR.sum()-sales_clean.Total_Value_INR.sum():,.0f}")

CLEANED DATASET
Purchase :   131 ->   131 rows (7 open POs flagged)
Store    :  1201 ->  1201 rows (4 quantities corrected)
Sales    :   407 ->   401 rows (6 duplicates removed)

Revenue as reported : Rs 42,254,152
Revenue after clean : Rs 41,361,460
Difference          : Rs 892,692


In [28]:
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 60)

issue_log = pd.DataFrame(issues)
order = {'High':0, 'Medium':1, 'Low':2, 'Info':3}
issue_log = (issue_log.assign(_s=issue_log.Severity.map(order))
                      .sort_values('_s').drop(columns='_s').reset_index(drop=True))
issue_log.index += 1

from IPython.display import display
display(issue_log[['Sheet','Check','Severity','Records','Issue']])

print(issue_log.Severity.value_counts().reindex(['High','Medium','Low','Info']).to_string())

,Sheet,Check,Severity,Records,Issue
1,Sales,Duplicate orders,High,6,6 orders entered twice under new IDs (IDs SO-0402 to SO-...
2,Sales,Inconsistent customer names,High,104,"4 customers entered under two spellings each (e.g. ""Summ..."
3,Store_Inventory,Ledger balance reconciliation,High,4,"4 of 1,183 rows fail the running-balance check: 2 sign r..."
4,Sales,Order_Status reliability,Medium,161,"161 orders marked ""Fulfilled"" were dispatched 1-3 days a..."
5,Purchase,Missing quality grade,Low,2,Quality_Grade blank on 2 POs
6,Purchase,Missing delivery dates,Info,7,7 POs have no Actual_Delivery_Date - all placed 17-30 Ju...
7,Sales,Missing dispatch dates,Info,29,29 orders have no Actual_Dispatch_Date - all are Pending...
8,Store_Inventory,Date range,Info,6,"Ledger extends to 05-Jul-2026, past the Jan-Jun window; ..."
9,Purchase + Sales,Arithmetic and date logic,Info,0,Total = Quantity x Rate verified on all rows; no deliver...
10,Store_Inventory,Referential integrity,Info,0,All PO and sales order links resolve to existing records...


Severity
High      3
Medium    1
Low       1
Info      5


In [29]:
from pathlib import Path
OUT = Path('../data/processed'); OUT.mkdir(exist_ok=True)

# CSVs for the SQL step
purchase_clean.to_csv(OUT / 'clean_purchase.csv', index=False)
store_clean.to_csv(OUT / 'clean_store_inventory.csv', index=False)
sales_clean.to_csv(OUT / 'clean_sales.csv', index=False)

# One workbook a manager can open without installing anything
with pd.ExcelWriter(OUT / 'cleaned_data_with_issue_log.xlsx', engine='openpyxl') as xl:
    issue_log.to_excel(xl, sheet_name='Data Quality Log', index=False)
    purchase_clean.to_excel(xl, sheet_name='Purchase (clean)', index=False)
    store_clean.to_excel(xl, sheet_name='Store (clean)', index=False)
    sales_clean.to_excel(xl, sheet_name='Sales (clean)', index=False)

print("Files written:")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:<32} {f.stat().st_size/1024:>8.1f} KB")

Files written:
  clean_purchase.csv                   18.1 KB
  clean_sales.csv                      61.5 KB
  clean_store_inventory.csv           129.9 KB
  cleaned_data_with_issue_log.xlsx    120.5 KB


## Chart data

Monthly series for the two headline findings, exported for Power BI and used for the charts in Step 2.

In [30]:
# Monthly series behind the two headline charts
store_clean['month'] = pd.to_datetime(store_clean.Txn_Date).dt.to_period('M').astype(str)
sales_clean['month'] = pd.to_datetime(sales_clean.Order_Date).dt.to_period('M').astype(str)

# --- Chart 1: the supply chain break ---
ss_closing = (store_clean[store_clean.Item_Code=='RM-SS']
              .sort_values(['Txn_Date','Txn_ID']).groupby('month').Balance_After_Txn.last()
              .rename('Steel_Scrap_Closing_Kg'))
ss_received = (store_clean[(store_clean.Item_Code=='RM-SS') &
               (store_clean.Movement_Type=='Inward-Purchase')]
               .groupby('month').Quantity.sum().rename('Steel_Scrap_Received_Kg'))
delayed = (sales_clean[sales_clean.Order_Status=='Delayed']
           .groupby('month').Order_ID.count().rename('Delayed_Orders'))

chart1 = pd.concat([ss_received, ss_closing, delayed], axis=1).fillna(0).reset_index()

# --- Chart 2: the seasonal mismatch ---
sugar = (sales_clean[sales_clean.Customer_Industry=='Sugar']
         .groupby('month').Total_Value_INR.sum().rename('Sugar_Order_Value_INR'))
pi_fm_closing = (store_clean[store_clean.Item_Code.isin(['RM-PI','RM-FM'])]
                 .sort_values(['Txn_Date','Txn_ID'])
                 .groupby(['month','Item_Code']).Balance_After_Txn.last()
                 .unstack().sum(axis=1).rename('PigIron_FerroMn_Closing_Kg'))
pi_fm_issued = (store_clean[(store_clean.Item_Code.isin(['RM-PI','RM-FM'])) &
                (store_clean.Movement_Type=='Issued-to-Production')]
                .groupby('month').Quantity.sum().rename('PigIron_FerroMn_Issued_Kg'))

chart2 = pd.concat([sugar, pi_fm_issued, pi_fm_closing], axis=1).fillna(0).reset_index()

with pd.ExcelWriter(OUT / 'chart_data.xlsx', engine='openpyxl') as xl:
    chart1.to_excel(xl, sheet_name='Chart1_Supply_Break', index=False)
    chart2.to_excel(xl, sheet_name='Chart2_Seasonal_Mismatch', index=False)

print(chart1.to_string(index=False))
print()
print(chart2.to_string(index=False))

  month  Steel_Scrap_Received_Kg  Steel_Scrap_Closing_Kg  Delayed_Orders
2026-01                     5780                    2439             0.0
2026-02                     6450                    2585            16.0
2026-03                     2233                     324            37.0
2026-04                     9656                    4077             0.0
2026-05                     5720                    2605             0.0
2026-06                     6542                    1360             0.0

  month  Sugar_Order_Value_INR  PigIron_FerroMn_Issued_Kg  PigIron_FerroMn_Closing_Kg
2026-01             2518964.32                       6897                        1915
2026-02             1903822.23                       6244                        1253
2026-03             1816005.42                       7719                        1412
2026-04              541398.93                       2314                        5070
2026-05               50969.90                       2499 